#### Hybrid Movie Recommendation System

This project develops a **hybrid movie recommendation system** that combines **collaborative filtering using matrix factorization and content-based filtering using cosine similarity** to generate personalize recommendations. The dataset was sourced from **GroupLens (MovieLens 1M)** and the workflow is as follows;
1. Problem Statement
2. Dataset Description
3. Data Loading
4. Data Preprocessing
5. Feature Engineering
6. Data Training
7. Model Evaluation
8. Deployment
9. Insights and Conclusion

##### **1. Problem Statement**

These days, platforms like Netflix and Amazon have thousands of movie options, and most users don't know what to pick or spends so much time looking for what to watch. So, platforms who doesn't recommends something relevant can make users get bored, confused or leave, which is a problem, so suggesting movies that match user's taste based on past behavaviours can be very helpful.

**The goal of this project is to build a hybrid movie recommendation system that combines filtering and content based filtering** to give better and more personalzed suggestions which can help reduce problems like sparsity and cold start while simulating how real-world streaming platforms personalize content for their users

##### **2. Dataset Description**

For this project, i used **MovieLens 1M dataset** collected from **GroupLens**. The dataset contains **1 million movie ratings** from about **6040 users** on around **3706 movies**. Each rating is on a **scale of 1 to 5**.

The dataset includes three main files:
* Users (with demographic details like age and gender)
* Movies (with titles and genres)
* Ratings (which link users to the movies they rated)

One major characteristics of this dataset is sparsity, since most users only rate small fraction of the total movies available which makes it good for testing recommendation systems, especially hybrids models designed to handle sparsity and cold start problems

##### **3. Dataset Loading**

In [2]:
# 3.1 loading the dataset
import pandas as pd
import numpy as np

# movies
movies = pd.read_csv('/content/movies.dat', sep='::', header=None, engine='python', encoding='latin-1', names=['MovieID', 'Title', 'Genres'])

# ratings
ratings = pd.read_csv('/content/ratings.dat', sep='::', header=None, engine='python', encoding='latin-1', names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

# users
users = pd.read_csv('/content/users.dat', sep='::', header=None, engine='python', encoding='latin-1', names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'])


In [3]:
# 3.2 basic inspection
print("Users:", ratings['UserID'].nunique())
print("Movies:", ratings['MovieID'].nunique())
print("Total Ratings:", ratings.shape[0])

print(ratings.head())
print(movies.head())

Users: 6040
Movies: 3706
Total Ratings: 1000209
   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy


In [4]:
# 3.3 checking for missing data
print(ratings.isnull().sum())
print(movies.isnull().sum())
print(users.isnull().sum())

UserID       0
MovieID      0
Rating       0
Timestamp    0
dtype: int64
MovieID    0
Title      0
Genres     0
dtype: int64
UserID        0
Gender        0
Age           0
Occupation    0
Zip-code      0
dtype: int64


##### **4. Data Preprocessing**

In [5]:
# 4.1 data preprocessing
# merging ratings with movies metadata as hybrid systems needs movies data together with ratings
data = ratings.merge(movies, on='MovieID', how='left')
print(data.head())

   UserID  MovieID  Rating  Timestamp                                   Title  \
0       1     1193       5  978300760  One Flew Over the Cuckoo's Nest (1975)   
1       1      661       3  978302109        James and the Giant Peach (1996)   
2       1      914       3  978301968                     My Fair Lady (1964)   
3       1     3408       4  978300275                  Erin Brockovich (2000)   
4       1     2355       5  978824291                    Bug's Life, A (1998)   

                         Genres  
0                         Drama  
1  Animation|Children's|Musical  
2               Musical|Romance  
3                         Drama  
4   Animation|Children's|Comedy  


In [6]:
# 4.2 creating user–item interaction matrix needed for collaborative filtering.
user_item_matrix = ratings.pivot(index='UserID', columns='MovieID', values='Rating')

In [7]:
# filling missing values with 0 for matrix factorization
R = user_item_matrix.fillna(0).values
num_users, num_movies = R.shape
print(f"Users: {num_users}, Movies: {num_movies}")

Users: 6040, Movies: 3706


In [8]:
print(movies.head())
print(movies.columns)

   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
Index(['MovieID', 'Title', 'Genres'], dtype='object')


##### **5. Feature Engineering**

In [9]:
# 4.3 encoding data(movie genres) for content based filtering
# spliting them first
movies['Genres'] = movies['Genres'].str.split('|')

# creating dummies variables
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(mlb.fit_transform(movies['Genres']), columns=mlb.classes_, index=movies['MovieID'])

print(genre_matrix.head())

         Action  Adventure  Animation  Children's  Comedy  Crime  Documentary  \
MovieID                                                                         
1             0          0          1           1       1      0            0   
2             0          1          0           1       0      0            0   
3             0          0          0           0       1      0            0   
4             0          0          0           0       1      0            0   
5             0          0          0           0       1      0            0   

         Drama  Fantasy  Film-Noir  Horror  Musical  Mystery  Romance  Sci-Fi  \
MovieID                                                                         
1            0        0          0       0        0        0        0       0   
2            0        1          0       0        0        0        0       0   
3            0        0          0       0        0        0        1       0   
4            1        0    

In [10]:
# 4.4 computing cosine similarities to create movie to movie similarity matrix
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(genre_matrix)


In [11]:
# 4.5 computing content-based recommendation function
def get_similar_movies(movie_id, top_n=10):
    idx = genre_matrix.index.get_loc(movie_id)
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices][['MovieID', 'Title']]

##### **6. Data Training (Numpy MF)**

In [12]:
# hyperparameters
k = 20           # latent features
steps = 50       # iterations
alpha = 0.002    # learning rate
beta = 0.02      # regularization

# initialize matrices
P = np.random.rand(num_users, k)
Q = np.random.rand(num_movies, k)

# training the loop
for step in range(steps):
    for i in range(num_users):
        for j in range(num_movies):
            if R[i, j] > 0:
                # Prediction
                pred = P[i, :].dot(Q[j, :].T)
                eij = R[i, j] - pred

                # Gradient descent update
                P[i, :] += alpha * (2 * eij * Q[j, :] - beta * P[i, :])
                Q[j, :] += alpha * (2 * eij * P[i, :] - beta * Q[j, :])

    # computing the total loss for every 10 steps
    if step % 10 == 0:
        pred_matrix = P.dot(Q.T)
        loss = 0
        for i in range(num_users):
            for j in range(num_movies):
                if R[i, j] > 0:
                    loss += (R[i, j] - pred_matrix[i, j]) ** 2
                    # regularization
                    loss += beta/2 * (np.sum(P[i, :]**2) + np.sum(Q[j, :]**2))
        print(f"Step {step}, loss: {loss:.4f}")


Step 0, loss: 1328145.3421
Step 10, loss: 836964.7506
Step 20, loss: 751772.6515
Step 30, loss: 704723.6988
Step 40, loss: 677275.4211


##### **7. Model Evaluation**

In [13]:
# computing the RMSE
from sklearn.metrics import mean_squared_error

pred_matrix = P.dot(Q.T)
mask = R > 0  # Only consider actual ratings
rmse = np.sqrt(mean_squared_error(R[mask], pred_matrix[mask]))
print(f"Collaborative Filtering RMSE: {rmse:.4f}")

Collaborative Filtering RMSE: 0.7429


##### **Hybrid Recommendation**

In [14]:
def hybrid_recommendation(user_id, movie_id, alpha=0.5):
    # Collaborative score
    collab_score = P[user_id-1, :].dot(Q[movie_id-1, :].T)

    # Content-based score
    idx = genre_matrix.index.get_loc(movie_id)
    content_score = cosine_sim[idx].mean()

    # Weighted hybrid
    final_score = alpha * collab_score + (1 - alpha) * content_score
    return final_score

In [15]:
print(hybrid_recommendation(user_id=5, movie_id=50))

1.865513985231749


###### **8. Model Deployment**

In [16]:
import pickle

# saving matrices
with open('P_matrix.pkl', 'wb') as f: pickle.dump(P, f)
with open('Q_matrix.pkl', 'wb') as f: pickle.dump(Q, f)
genre_matrix.to_pickle('genre_matrix.pkl')


In [17]:
# loading for inference
with open('P_matrix.pkl', 'rb') as f: P_loaded = pickle.load(f)
with open('Q_matrix.pkl', 'rb') as f: Q_loaded = pickle.load(f)
genre_matrix_loaded = pd.read_pickle('genre_matrix.pkl')
cosine_sim_loaded = cosine_similarity(genre_matrix_loaded)

In [18]:
def content_based_recommend(movie_id, top_n=10):
    idx = genre_matrix_loaded.index.get_loc(movie_id)
    sim_scores = list(enumerate(cosine_sim_loaded[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices][['MovieID', 'Title']]

def hybrid_recommendation_api(user_id, movie_id, alpha=0.5):
    collab_score = P_loaded[user_id-1, :].dot(Q_loaded[movie_id-1, :].T)
    idx = genre_matrix_loaded.index.get_loc(movie_id)
    content_score = cosine_sim_loaded[idx].mean()
    final_score = alpha * collab_score + (1 - alpha) * content_score
    return final_score

##### **9. Insights & Conclusion**

 - NumPy matrix factorization works without external dependencies
 - Content-based filtering improves cold-start recommendations
 - Hybrid balances user taste and movie similarity
 - RMSE shows predictive accuracy
 - Precomputed top-N recommendations simulate fast production deployment